# 4 — Guarded layouts: pad and stencil

Step 2's extension, now with charts. A **guard** is a linear form over
LATTICE coordinates with half-open bounds; violating coordinates resolve to
no location and the tensor supplies a **fill**. Charts sit strictly above:
physical coordinates normalize to lattice ints at the boundary, and guards
never see a Quantity — the compiler-facing algebra is untouched.

Working example: a 5-point stage scan at 25 nm pitch, values 1..5.

In [1]:
import numpy as np
from nbhelp import show  # also puts tensorlib on sys.path
from tensorlib import Tensor, q

In [2]:
arr = np.arange(1, 6, dtype=np.int64)
t = Tensor.from_numpy(arr, ("x",)).with_charts(x=("0 nm", "25 nm"))
show(t, "the base tensor")

-- the base tensor
Tensor[int64] on Buffer(40B @ cpu)
  offset : 0 bytes
  dim     stride  start   stop   size  chart
  x            8      0      5      5  pos[x: 0 nm step 25 nm]
  numel=5  footprint=(0, 40)  injectivity=injective


## pad — the dual of slice, with physical extents

The box grows beyond the mapped region; a guard remembers where the real
data lives; the chart is unchanged (the labeling extends naturally).

In [3]:
p = t.pad(fill=-1, x=(q("-50 nm"), q("175 nm")))
show(p, "pad x to [-50 nm, 175 nm)  — lattice [-2, 7)")
p.to_numpy()

-- pad x to [-50 nm, 175 nm)  — lattice [-2, 7)
Tensor[int64] on Buffer(40B @ cpu)
  offset : 0 bytes
  dim     stride  start   stop   size  chart
  x            8     -2      7      9  pos[x: 0 nm step 25 nm]
  numel=9  footprint=(0, 40)  injectivity=injective
  Guard(0 <= x < 5)
  fill   : -1


array([-1, -1,  1,  2,  3,  4,  5, -1, -1])

In [4]:
print("x=-25 nm resolves to:", p.layout.resolve(x=q("-25 nm")))
print("x=0 nm   resolves to:", p.layout.resolve(x=q("0 nm")))

x=-25 nm resolves to: None
x=0 nm   resolves to: 0


The **footprint honors the guard**: the box alone would reach byte
`8·(-2) = -16`, but the guard's bounds clamp the address contribution
exactly, so `check()` passes.

In [5]:
print(p.footprint())
p.check()
print("in bounds")

(0, 40)
in bounds


## simplify — guards discharge by interval arithmetic

Slice back to the interior (physically) and the guard becomes vacuous:
`simplify` drops it and the layout **collapses back to the core affine
family**.

In [6]:
interior = p.slice(x=(q("0 nm"), q("125 nm"))).simplify()
print(type(interior.layout).__name__)
show(interior, "interior slice, simplified")

Layout
-- interior slice, simplified
Tensor[int64] on Buffer(40B @ cpu)
  offset : 0 bytes
  dim     stride  start   stop   size  chart
  x            8      0      5      5  pos[x: 0 nm step 25 nm]
  numel=5  footprint=(0, 40)  injectivity=injective


In [7]:
edge = p.slice(x=(q("125 nm"), q("175 nm")))  # pure padding
print("always_fill:", edge.layout.always_fill())
edge.to_numpy()

always_fill: True


array([-1, -1])

## stencil — physically labeled taps with a boundary

`s[x=X, x_k=K] = t[x=X+K]`, or fill when the tap leaves the box. The kernel
range is INCLUSIVE (D1 sugar) and may be given in physical displacements;
`x_k` gets a displacement chart with x's step. The guard is on the lattice
form `x + x_k`.

In [8]:
s = t.stencil("x", k=(q("-25 nm"), q("25 nm")), fill=0)
show(s, "3-tap stencil at ±25 nm")
s.to_numpy()

-- 3-tap stencil at ±25 nm
Tensor[int64] on Buffer(40B @ cpu)
  offset : 0 bytes
  dim     stride  start   stop   size  chart
  x            8      0      5      5  pos[x: 0 nm step 25 nm]
  x_k          8     -1      2      3  disp[x: 0 nm step 25 nm]
  numel=15  footprint=(0, 40)  injectivity=unknown
  Guard(0 <= x + x_k < 5)
  fill   : 0


array([[0, 1, 2],
       [1, 2, 3],
       [2, 3, 4],
       [3, 4, 5],
       [4, 5, 0]])

In [9]:
print("left edge, left tap: ", s.item(x=q("0 nm"), x_k=q("-25 nm")))
print("interior tap:        ", s.item(x=q("50 nm"), x_k=q("25 nm")), "==", arr[3])

left edge, left tap:  0
interior tap:         4 == 4


## the interior of a stencil *is* a window

Slice the boundary away (physically), simplify, and the guard discharges —
leaving exactly notebook 3's affine sliding window. Peel the boundary, keep
the pure affine interior: the compiler story in miniature.

In [10]:
core = s.slice(x=(q("25 nm"), q("100 nm"))).simplify()
print(type(core.layout).__name__)
print(bool(np.array_equal(core.to_numpy(), t.window("x", "x_k", (-1, 2)).to_numpy())))

Layout
True


## guard rewrites with physical coordinates

`select` substitutes into the form; `shift` moves the bounds with the lattice
relabeling — and since the chart glues physics to data, the *physical*
boundary is invariant under shift.

In [11]:
row = s.select(x=q("0 nm"))
show(row, "select(x=0 nm): guard is now on x_k alone")
print(row.item(x_k=q("-25 nm")), row.item(x_k=q("0 nm")), row.item(x_k=q("25 nm")))

-- select(x=0 nm): guard is now on x_k alone
Tensor[int64] on Buffer(40B @ cpu)
  offset : 0 bytes
  dim     stride  start   stop   size  chart
  x_k          8     -1      2      3  pos[x: 0 nm step 25 nm]
  numel=3  footprint=(0, 16)  injectivity=injective
  Guard(0 <= x_k < 5)
  fill   : 0
0 1 2


In [12]:
moved = s.shift(x=q("250 nm"))  # lattice +10; physics glued
print("boundary still at x=0 nm:", moved.item(x=q("0 nm"), x_k=q("-25 nm")) == 0)

boundary still at x=0 nm: True


## split through a stencil guard (compiler mode)

Blocking substitutes `x = 4·xo + xi` into the guard — the coefficient on
`xo` becomes 4, and boundary detection keeps working across the blocking.
Lattice ints remain first-class throughout.

In [13]:
t8 = Tensor.from_numpy(np.arange(1, 9, dtype=np.int64), ("x",))
t8 = t8.with_charts(x=("0 nm", "25 nm"))
s8 = t8.stencil("x", k=(-1, 1), fill=0)
print("before:", s8.layout.guards)
blocked = s8.split("x", xo=2, xi=4)
print("after: ", blocked.layout.guards)
print(blocked.item(xo=0, xi=0, x_k=-1), blocked.item(xo=1, xi=0, x_k=-1))

before: (Guard(0 <= x + x_k < 8),)
after:  (Guard(0 <= x_k + xi + 4*xo < 8),)
0 4


## dilation — atrous taps, still linear

`dilation=2` puts taps two lattice steps apart: the kernel stride doubles,
the guard form becomes `x + 2·x_k`, and the tap chart's step is the dilated
pitch (50 nm here). The white-box algebra doesn't blink.

In [14]:
sd = t.stencil("x", k=(-1, 1), fill=0, dilation=2)
show(sd, "dilated 3-tap stencil at ±50 nm")
print(sd.item(x=q("50 nm"), x_k=q("-50 nm")), "==", arr[0])
print("boundary:", sd.item(x=q("25 nm"), x_k=q("-50 nm")))

-- dilated 3-tap stencil at ±50 nm
Tensor[int64] on Buffer(40B @ cpu)
  offset : 0 bytes
  dim     stride  start   stop   size  chart
  x            8      0      5      5  pos[x: 0 nm step 25 nm]
  x_k         16     -1      2      3  disp[x: 0 nm step 50 nm]
  numel=15  footprint=(0, 40)  injectivity=unknown
  Guard(0 <= x + 2*x_k < 5)
  fill   : 0
1 == 1
boundary: 0


## guards compose with the whole op set

Flip, diagonal, window, decimate, and (conditionally) merge now rewrite
guards too. Two favorites: flipping a padded tensor just reverses the
storage — fills and all; and the diagonal of a padded matrix reads fill
outside the real region, which composed with shift gives banded views.

In [15]:
print(p.flip("x").to_numpy(), "  (the pad, reversed)")

[-1 -1  5  4  3  2  1 -1 -1]   (the pad, reversed)


In [16]:
img = Tensor.from_numpy(np.arange(16, dtype=np.int64).reshape(4, 4), ("i", "j"))
pim = img.pad(fill=-1, i=(-1, 5), j=(-1, 5))
main = pim.diagonal(("i", "j"), "z")
sup = pim.shift(j=-1).diagonal(("i", "j"), "z")  # superdiagonal, falls off the edge
print("main diagonal: ", main.to_numpy())
print("superdiagonal: ", sup.to_numpy())

main diagonal:  [-1  0  5 10 15 -1]
superdiagonal:  [-1  1  6 11 -1]


## composing pad and stencil

Stenciling a *padded* tensor must see the padding as fill: `stencil`
substitutes `x → x + x_k` into existing guards too — watch the pad guard
gain an `x_k` coefficient.

In [17]:
t4 = Tensor.from_numpy(np.arange(1, 5, dtype=np.int64), ("x",))
t4 = t4.with_charts(x=("0 nm", "25 nm"))
ps = t4.pad(fill=0, x=(q("0 nm"), q("150 nm"))).stencil("x", k=(q("0 nm"), q("25 nm")), fill=0)
for g in ps.layout.guards:
    print(g)
print("real tap:", ps.item(x=q("75 nm"), x_k=q("0 nm")), "  padded tap:", ps.item(x=q("75 nm"), x_k=q("25 nm")))

Guard(0 <= x + x_k < 4)
Guard(0 <= x + x_k < 6)
real tap: 4   padded tap: 0


---
That is the whole library: an affine map + box domain, one guard extension,
and an exact labeling layer (units + charts) over an unchanged lattice.
Every operation is a small algebraic rewrite of that representation. Tests
in `../tests/` pin each rewrite against numpy ground truth; the design
rationale lives in `../DESIGN.md`, the open edges in `../CONCERNS.md`.